# PSE-823: Advanced Process Dynamics and Control
## Lecture 1 -- Introduction: Control Fundamentals and the DCS Implementation Layer
### (Coughanowr & LeBlanc, Ch. 1; Cecil Smith, Introduction and Sec 1.1-1.2)

Date: __________

## Agenda

**Part A -- Course Map**
1. Two textbooks, two jobs
2. The 15-week arc

**Part B -- Economics and the DCS Block Paradigm** (Cecil, Intro & Sec 1.1-1.2)
3. Narrow the variance, shift the target
4. Blocks, tags, and the P&ID
5. Bumpless transfer and windup

**Part C -- Control System Fundamentals** (Coughanowr Ch. 1)
6. Control vocabulary and feedback
7. On/off, P, and PI control; offset
8. A first warning about stability

## Part A
### Course Map

(course overview -- no assigned textbook chapter)
<!-- source: course-overview -->

### Concept

Two textbooks, two jobs, no overlap:

- **Coughanowr & LeBlanc** -- the spine. Derivations, transfer functions, stability, frequency response, state-space.
- **Cecil Smith** -- the industrial layer. Cascade, override, split-range, DCS structure, and this course's only source for MPC.

### Concept

Fifteen weeks: about eight of classical theory (Coughanowr), building to frequency response and a natural midterm; then advanced strategies, state-space, nonlinear systems, and MPC.

This is my first offering of PSE-823 -- weeks overlapping CHE-323 move fast, as review.

### Python: course structure as data

No chapter math yet -- a running theme starts instead: every lecture keeps a small Python snippet on-screen. Today's is just the roadmap itself.

In [1]:
weeks = {
    1: ("Coughanowr Ch.1 + Cecil Intro/1.1-1.2", "review"),
    2: ("Coughanowr Ch.2-3", "new"),
    9: ("Cecil Ch.2-7", "new"),
    15: ("Cecil Sec 8.3 (MPC)", "new"),
}
for wk, (source, flag) in weeks.items():
    print(f"Week {wk:>2}: {source:<32} [{flag}]")

Week  1: Coughanowr Ch.1 + Cecil Intro/1.1-1.2 [review]
Week  2: Coughanowr Ch.2-3                [new]
Week  9: Cecil Ch.2-7                     [new]
Week 15: Cecil Sec 8.3 (MPC)              [new]


### Check

Discussion: for "why does a plant use override control instead of just tuning harder," which book answers it, and why?

Expected: Cecil, not Coughanowr -- a plant-structure/practice question, not a derivation.

## Part B
### Economics and the DCS Block Paradigm

(Cecil Smith, Introduction and Sec 1.1-1.2)
<!-- source: cecil-intro-1.1-1.2 -->

### Concept

Most control needs are met by simple feedback: a measurement, a PID, a valve. Performance = **variance in the control error** (SP minus PV).

Cecil's phrase: **"narrow the variance, shift the target."**

### Concept

Two ways to act on the economic incentive:

- **Replace the PID** -- usually MPC. Powerful, but needs real economic benefit to justify.
- **Retain the PID, add logic around it** -- DCS function blocks. Cheaper, more common.

This fork is the whole course: Coughanowr strengthens the PID; Cecil Ch.2-7 adds logic; Sec 8.3 replaces it with MPC.

### Concept

Real systems are built from **blocks**:

- **Input/measurement block** -- field signal to engineering units
- **Output/valve block** -- controller output to valve command (fail-open/closed)
- **Control block** -- the algorithm; PID is the most common

Every block output: **`<Tag>.<Attribute>`**, e.g. `TC4011.SP`.

### Example

**Figure 1.1 -- P&ID of a level-to-flow cascade.** The level controller's output becomes the flow controller's **remote set point (RSP)**.

![](images/fig1-1_cascade_pid.jpeg)

Your first **cascade** -- one controller driving another's set point (Week 8).

### Concept

Two problems a clean block diagram glosses over:

**Bumpless transfer** -- switching manual/auto/cascade must not "bump" the process.

**Windup** (Cecil's definition): *"Reset windup occurs when changes in controller output have no effect on the process variable."*

### Example

**Figure 1.2 -- control logic diagram**, same cascade, showing the extra softwired connections for bumpless transfer/windup protection.

![](images/fig1-2_cascade_control_logic.jpeg)

If the discharge valve is fully open, the *level* controller is the one that winds up, even though the *flow* controller's valve saturated. Fixes: integral tracking, external reset, inhibit increase/decrease.

### Concept

A fast tour -- recognize these, don't memorize:

- **PID controller**
- **Arithmetic**: summer, multiplier, divider, characterization function
- **Logic gates**: NOT/OR/AND/XOR, comparator, one-shot
- **Dynamic**: integrator, lead-lag, dead time, moving average
- **Special**: selector (Week 9's override), cutoff, hand station

### Python

A DCS summer block is just $Y=k_0+k_1X_1+k_2X_2$. We can check a windup condition the same way: has the block's output stopped moving the process?

In [2]:
def summer_block(k0, k1, k2, x1, x2):
    return k0 + k1 * x1 + k2 * x2

def windup_condition(valve_pct, output_change):
    at_limit = valve_pct >= 100 or valve_pct <= 0
    return at_limit and output_change != 0

fc_output = summer_block(0, 1, 0, x1=85.0, x2=0)
print(fc_output, windup_condition(100, output_change=3.0))

85.0 True


### Your Turn (8 min)

Sketch the block-and-tag diagram (P&ID style) for a **cascade temperature-to-steam-flow control** on a heat exchanger: TC's output is FC's set point.

Label: TC.PV, TC output, FC.PV, FC.RSP, FC output. Under what condition does the *inner* loop need to send a windup-tracking signal to the *outer* loop?

TC.PV = tube-side outlet temperature transmitter
TC.MN (output) -> FC.RSP
FC.PV = steam flow transmitter
FC.MN (output) -> steam valve opening
Windup condition: steam valve fully open/closed -- further TC output changes can't change steam flow, so TC needs a track/inhibit signal from FC, mirroring the level-to-flow example.

### Check

Cold-call for the tag.attribute chain, then: why does the *inner* loop know when the valve has saturated, not the outer one?

The inner loop's PV/output relationship is what actually touches the valve -- the outer loop only sees a set point it handed off.

## Part C
### Control System Fundamentals

(Coughanowr & LeBlanc, Ch. 1)
<!-- source: coughanowr-ch1 -->

### Concept

Automatic control pays for itself: safety, environmental limits, product quality, efficient resource use, profitability.

A control system holds a **controlled variable** at its **set point** by adjusting a **manipulated variable**, despite **disturbances**: disturbance rejection plus set point tracking.

### Example

**Cruise control**: controlled var = speed; set point = dialed speed; manipulated var = throttle; disturbances = hills, wind.

**Thermostat**: controlled var = room temp; manipulated var = furnace input; disturbances = heat loss, doors.

### Concept

$$\text{Error} = (\text{Set point}) - (\text{Measured value})$$

**Figure 1-1 -- generalized block diagram**

![](images/fig1-3_generalized_block_diagram.jpeg)

**Feedback control**: deviation info feeds back. Closed-loop = automatic; open-loop = manual. Negative feedback dominates practice; positive feedback tends toward instability.

### Example -- Example 1.1, hot water tank

Set point $130^\circ F$. Error $>0$: open fuel valve; Error $\le 0$: close it. Disturbances: ambient loss, hot-water demand.

![](images/fig1-4_hot_water_tank_a.jpeg)
![](images/fig1-5_hot_water_tank_b.jpeg)

### Concept

On/off is full-on or full-off. **Proportional control** scales heat input by the error via **gain**: larger gain gives smaller steady-state error, but P-only never reaches zero error -- **offset** remains.

**Integral control** added to P drives offset to exactly zero, at the price of more oscillation.

### Concept -- a complication the book flags immediately

The controller sees a lagged measurement, not the true value -- it acts on **apparent error**. Lag makes the response more oscillatory, and enough gain can make the oscillation **grow**: instability.

The book's warning: any amount of integral action can cause instability in some systems -- why Ch.13 onward formalizes this.

### Python

The gain-vs-offset tradeoff has a closed form derived properly starting Ch.7: $e_{ss}=\dfrac{\Delta d}{1+K_cK_p}$. `sympy` lets us see the limit as $K_c\to\infty$ symbolically.

In [3]:
import sympy as sp

Kc, Kp, d = sp.symbols('K_c K_p Delta_d', positive=True)
e_ss = d / (1 + Kc * Kp)
limit_Kc_inf = sp.limit(e_ss, Kc, sp.oo)
print(e_ss, "-> as Kc grows:", limit_Kc_inf)

Delta_d/(K_c*K_p + 1) -> as Kc grows: 0


### Your Turn (10 min)

Two identical hot-water systems get the same $10^\circ F$ disturbance. System A: P-only, low gain. System B: P-only, high gain.

Sketch the qualitative response for each -- which has smaller offset, which is more likely unstable with enough measurement lag?

Then draw the block diagram (style of Fig. 1-1) for **Problem 1.3**: an automobile cruise control system.

Higher gain -> smaller offset, but more oscillatory, and with enough measurement lag, high gain gives growing-amplitude (unstable) oscillation.
Cruise control: controlled var = vehicle speed; manipulated var = throttle/fuel flow; disturbances = hills/wind/terrain; measurement = speedometer; controller = cruise module.

### Check

Cold-call one student on the gain comparison, a second on the cruise-control diagram. If we disconnect the feedback signal, is it still a control system?

Yes -- but open-loop/manual mode, per the book's own definition.

### Wrap-up

**Covered today:**
- Two-textbook structure and the 15-week arc
- Cecil's economics, blocks/tags, bumpless transfer, windup
- Coughanowr's vocabulary, feedback, P/PI control, offset, the stability warning

**Next class:** Chapter 2 (modeling) and Chapter 3 (partial fractions) -- turning today's qualitative warning into something you can solve.